# 08 — Codegen Analysis


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from build_optimiser.config import Config
from build_optimiser.graph import load_graph, attach_metrics
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

cfg = Config.from_yaml('../config.yaml')
file_df = pd.read_parquet('../data/processed/file_metrics.parquet')
target_df = pd.read_parquet('../data/processed/target_metrics.parquet')
edge_df = pd.read_parquet('../data/processed/edge_list.parquet')
G = load_graph('../data/processed/edge_list.parquet')
attach_metrics(G, target_df)

%matplotlib inline
sns.set_theme(style='whitegrid')

## Codegen Inventory Summary

Total generated files, SLOC, compile time, % of total build cost.


In [ ]:
# --- Codegen Inventory Summary ---

def pct(part, whole):
    return f"{part / whole:.1%}" if whole > 0 else "N/A"

gen_files = file_df[file_df['is_generated']]
authored_files = file_df[~file_df['is_generated']]
codegen_targets = target_df[target_df['codegen_file_count'] > 0].copy()

# Scalars
total_files = len(file_df)
total_gen = len(gen_files)
total_authored = len(authored_files)

total_sloc = int(target_df['code_lines_total'].sum())
gen_sloc = int(target_df['code_lines_generated'].sum())

total_compile_ms = int(target_df['compile_time_sum_ms'].sum())
gen_compile_ms = int(target_df['codegen_compile_time_sum_ms'].sum())

total_build_ms = int(target_df['total_build_time_ms'].sum())
total_codegen_step_ms = int(target_df['codegen_time_ms'].fillna(0).sum())

w = 70
print("=" * w)
print(f"{'Metric':<35} {'Total':>10} {'Codegen':>10} {'%':>8}")
print("-" * w)
print(f"{'Source files':<35} {total_files:>10,} "
      f"{total_gen:>10,} {pct(total_gen, total_files):>8}")
print(f"{'SLOC':<35} {total_sloc:>10,} "
      f"{gen_sloc:>10,} {pct(gen_sloc, total_sloc):>8}")
print(f"{'Compile time (ms)':<35} {total_compile_ms:>10,} "
      f"{gen_compile_ms:>10,} {pct(gen_compile_ms, total_compile_ms):>8}")
print(f"{'Codegen step time (ms)':<35} {total_build_ms:>10,} "
      f"{total_codegen_step_ms:>10,} {pct(total_codegen_step_ms, total_build_ms):>8}")
print(f"{'Targets with codegen':<35} {len(target_df):>10,} "
      f"{len(codegen_targets):>10,} {pct(len(codegen_targets), len(target_df)):>8}")
print("=" * w)

# --- Per-target breakdown ---
codegen_targets = codegen_targets.sort_values(
    'codegen_compile_time_sum_ms', ascending=False,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.barh(codegen_targets['cmake_target'],
         codegen_targets['codegen_file_count'], color='darkorange')
ax1.barh(codegen_targets['cmake_target'],
         codegen_targets['authored_file_count'],
         left=codegen_targets['codegen_file_count'], color='steelblue')
ax1.set_xlabel('File count')
ax1.set_title('Files per target (generated vs authored)')
ax1.legend(['Generated', 'Authored'], loc='lower right')
ax1.invert_yaxis()

ax2.barh(codegen_targets['cmake_target'],
         codegen_targets['codegen_compile_time_sum_ms'], color='darkorange')
ax2.barh(codegen_targets['cmake_target'],
         codegen_targets['authored_compile_time_sum_ms'],
         left=codegen_targets['codegen_compile_time_sum_ms'],
         color='steelblue')
ax2.set_xlabel('Compile time (ms)')
ax2.set_title('Compile time per target (generated vs authored)')
ax2.legend(['Generated', 'Authored'], loc='lower right')
ax2.invert_yaxis()

fig.suptitle('Codegen Inventory — Per-Target Breakdown', fontsize=14)
plt.tight_layout()
plt.show()

# --- Donut charts ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].pie(
    [total_gen, total_authored], labels=['Generated', 'Authored'],
    colors=['darkorange', 'steelblue'], autopct='%1.1f%%',
    startangle=90, wedgeprops={'width': 0.5},
)
axes[0].set_title('Files')

axes[1].pie(
    [gen_sloc, total_sloc - gen_sloc], labels=['Generated', 'Authored'],
    colors=['darkorange', 'steelblue'], autopct='%1.1f%%',
    startangle=90, wedgeprops={'width': 0.5},
)
axes[1].set_title('SLOC')

axes[2].pie(
    [gen_compile_ms, total_compile_ms - gen_compile_ms],
    labels=['Generated', 'Authored'],
    colors=['darkorange', 'steelblue'], autopct='%1.1f%%',
    startangle=90, wedgeprops={'width': 0.5},
)
axes[2].set_title('Compile Time')

fig.suptitle('Codegen Share of Build', fontsize=14)
plt.tight_layout()
plt.show()

## Codegen Timing Analysis

Ninja log codegen step durations. Which generators are slowest?


In [ ]:
# --- Codegen Timing Analysis ---
from matplotlib.patches import Patch

ninja_log = pd.read_csv(str(cfg.raw_data_dir / 'ninja_log.csv'))
build_steps = ninja_log[ninja_log['step_type'] != 'other'].copy()

step_colors = {'compile': 'steelblue', 'codegen': 'darkorange', 'archive': '#55A868', 'link': '#C44E52'}

# --- Time split by step type ---
time_by_type = build_steps.groupby('step_type')['duration_ms'].sum().sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

colors = [step_colors.get(t, 'grey') for t in time_by_type.index]
ax1.bar(time_by_type.index, time_by_type.values, color=colors)
ax1.set_ylabel('Total duration (ms)')
ax1.set_title('Wall-clock time by build step type')
for i, (typ, val) in enumerate(time_by_type.items()):
    ax1.text(i, val + time_by_type.max() * 0.01, f'{val:,.0f}ms', ha='center', fontsize=9)

# --- Codegen steps detail ---
codegen_steps = build_steps[build_steps['step_type'] == 'codegen'].sort_values('duration_ms', ascending=False)

if len(codegen_steps) > 0:
    ax2.barh(codegen_steps['cmake_target'] + '\n' + codegen_steps['output_path'].str.split('/').str[-1],
             codegen_steps['duration_ms'], color='darkorange')
    ax2.set_xlabel('Duration (ms)')
    ax2.set_title('Codegen step durations')
    ax2.invert_yaxis()
else:
    ax2.text(0.5, 0.5, 'No codegen steps found', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('Codegen step durations')

fig.suptitle('Codegen Timing Analysis', fontsize=14)
plt.tight_layout()
plt.show()

print("=== Codegen Steps ===")
if len(codegen_steps) > 0:
    display(codegen_steps[['cmake_target', 'output_path', 'start_ms', 'end_ms', 'duration_ms']].reset_index(drop=True))
else:
    print("No codegen steps in ninja log.")

# --- Build timeline with codegen highlighted ---
target_order = build_steps.groupby('cmake_target')['start_ms'].min().sort_values().index.tolist()
y_map = {t: i for i, t in enumerate(target_order)}

fig, ax = plt.subplots(figsize=(16, max(6, len(target_order) * 0.4)))

for _, row in build_steps.iterrows():
    y = y_map.get(row['cmake_target'], 0)
    ax.barh(y, row['end_ms'] - row['start_ms'], left=row['start_ms'],
            height=0.7, color=step_colors.get(row['step_type'], 'grey'), alpha=0.85)

ax.set_yticks(list(y_map.values()))
ax.set_yticklabels(list(y_map.keys()), fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Time (ms)')
ax.set_title('Build Timeline (codegen steps highlighted)', fontsize=14)
ax.legend(handles=[Patch(color=c, label=t) for t, c in step_colors.items()], loc='lower right')
plt.tight_layout()
plt.show()

## Codegen Fan-Out Analysis

How many targets consume each codegen output?


In [ ]:
# --- Codegen Fan-Out Analysis ---

codegen_tgts = target_df[target_df['codegen_file_count'] > 0].copy()
rev_G = G.reverse()

rows = []
for _, row in codegen_tgts.iterrows():
    t = row['cmake_target']
    if t not in G:
        continue
    direct_deps = [n for n in G.predecessors(t) if G[n][t].get('is_direct', False)]
    transitive_deps = nx.descendants(rev_G, t) - set(direct_deps)
    rows.append({
        'cmake_target': t,
        'codegen_time_ms': int(row.get('codegen_time_ms') or 0),
        'codegen_compile_time_sum_ms': int(row.get('codegen_compile_time_sum_ms') or 0),
        'direct_dependants': len(direct_deps),
        'transitive_dependants': len(transitive_deps),
        'total_dependants': len(direct_deps) + len(transitive_deps),
    })

fanout_df = pd.DataFrame(rows).sort_values('total_dependants', ascending=False)

print("=== Codegen Fan-Out ===")
display(fanout_df.reset_index(drop=True))

# --- Stacked bar: direct vs transitive dependants ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, max(4, len(fanout_df) * 0.5)))

ax1.barh(fanout_df['cmake_target'], fanout_df['direct_dependants'], color='steelblue', label='Direct')
ax1.barh(fanout_df['cmake_target'], fanout_df['transitive_dependants'],
         left=fanout_df['direct_dependants'], color='darkorange', label='Transitive')
ax1.set_xlabel('Number of dependant targets')
ax1.set_title('Codegen fan-out (direct + transitive)')
ax1.legend(loc='lower right')
ax1.invert_yaxis()

# --- Scatter: codegen time vs dependant count ---
if len(fanout_df) > 0:
    ax2.scatter(fanout_df['codegen_time_ms'], fanout_df['total_dependants'],
                s=fanout_df['codegen_compile_time_sum_ms'].clip(lower=20) / 5,
                c='darkorange', alpha=0.7, edgecolors='black', linewidths=0.5)
    for _, r in fanout_df.iterrows():
        ax2.annotate(r['cmake_target'], (r['codegen_time_ms'], r['total_dependants']),
                     fontsize=8, ha='left', va='bottom')
    ax2.set_xlabel('Codegen step time (ms)')
    ax2.set_ylabel('Total dependant targets')
    ax2.set_title('Codegen time vs fan-out\n(bubble size = codegen compile cost)')

fig.suptitle('Codegen Fan-Out Analysis', fontsize=14)
plt.tight_layout()
plt.show()

## Generated File Compilation Cost

Compare compile time, preprocessed size, header depth: generated vs authored.


In [ ]:
# --- Generated File Compilation Cost ---

compilable = file_df[file_df['compile_time_ms'] > 0].copy()

metrics_to_compare = [
    ('compile_time_ms', 'Compile time (ms)', True),
    ('preprocessed_bytes', 'Preprocessed size (bytes)', True),
    ('header_max_depth', 'Header max depth', False),
    ('expansion_ratio', 'Expansion ratio', False),
]

palette = {False: 'steelblue', True: 'darkorange'}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (col, label, use_log) in zip(axes.flat, metrics_to_compare):
    plot_data = compilable[compilable[col] > 0] if use_log else compilable.dropna(subset=[col])
    if len(plot_data) == 0:
        ax.text(0.5, 0.5, f'No data for {col}', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(label)
        continue
    sns.histplot(data=plot_data, x=col, hue='is_generated', palette=palette,
                 stat='density', alpha=0.6, bins=20, log_scale=use_log, ax=ax, legend=True)
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.set_title(label)

fig.suptitle('Generated vs Authored File Compilation Metrics', fontsize=14)
plt.tight_layout()
plt.show()

# --- Summary statistics table ---
authored_comp = compilable[~compilable['is_generated']]
gen_comp = compilable[compilable['is_generated']]

stat_cols = ['compile_time_ms', 'preprocessed_bytes', 'header_max_depth', 'expansion_ratio']
summary_rows = []
for col in stat_cols:
    summary_rows.append({
        'Metric': col,
        'Authored mean': authored_comp[col].mean() if len(authored_comp) > 0 else 0,
        'Authored median': authored_comp[col].median() if len(authored_comp) > 0 else 0,
        'Generated mean': gen_comp[col].mean() if len(gen_comp) > 0 else 0,
        'Generated median': gen_comp[col].median() if len(gen_comp) > 0 else 0,
    })

print("=== Compilation Cost: Generated vs Authored ===")
summary = pd.DataFrame(summary_rows)
for c in summary.columns[1:]:
    summary[c] = summary[c].map(lambda x: f'{x:,.1f}')
display(summary.reset_index(drop=True))

# --- Worst offenders: top generated files by compile time ---
if len(gen_comp) > 0:
    worst = gen_comp.nlargest(10, 'compile_time_ms')[
        ['source_file', 'cmake_target', 'compile_time_ms', 'preprocessed_bytes',
         'header_max_depth', 'expansion_ratio', 'code_lines']
    ].copy()
    worst['source_file'] = worst['source_file'].str.split('/').str[-1]
    print("\n=== Top 10 Generated Files by Compile Time ===")
    display(worst.reset_index(drop=True))
else:
    print("\nNo compilable generated files found.")

## Codegen-Triggered Rebuild Analysis

Total rebuild cost per codegen step x change frequency.


In [ ]:
# --- Codegen-Triggered Rebuild Analysis ---
from build_optimiser.simulation import codegen_cascade_cost

codegen_tgts = target_df[target_df['codegen_file_count'] > 0].copy()
working_days = cfg.git_history_months * 20
rev_G = G.reverse()

rows = []
for _, row in codegen_tgts.iterrows():
    t = row['cmake_target']
    if t not in G:
        continue
    cascade = codegen_cascade_cost(G, t, target_df)
    commit_count = int(row.get('git_commit_count_total') or 0)
    change_prob = min(commit_count / working_days, 1.0) if working_days > 0 else 0.0
    rows.append({
        'cmake_target': t,
        'codegen_time_ms': int(row.get('codegen_time_ms') or 0),
        'cascade_cost_ms': cascade,
        'change_prob': change_prob,
        'expected_daily_rebuild_ms': change_prob * cascade,
        'dependant_count': len(nx.descendants(rev_G, t)),
    })

cascade_df = pd.DataFrame(rows).sort_values('expected_daily_rebuild_ms', ascending=False)

print("=== Codegen-Triggered Rebuild Cost ===")
display(cascade_df.reset_index(drop=True))

# --- Horizontal bar: cascade cost and expected daily cost ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, max(4, len(cascade_df) * 0.5)))

ax1.barh(cascade_df['cmake_target'], cascade_df['cascade_cost_ms'], color='darkorange')
ax1.set_xlabel('Total cascade rebuild cost (ms)')
ax1.set_title('Rebuild cost if codegen output changes')
ax1.invert_yaxis()

ax2.barh(cascade_df['cmake_target'], cascade_df['expected_daily_rebuild_ms'], color='crimson')
ax2.set_xlabel('Expected daily rebuild cost (ms)')
ax2.set_title('Cascade cost × change probability')
ax2.invert_yaxis()

fig.suptitle('Codegen-Triggered Rebuild Analysis', fontsize=14)
plt.tight_layout()
plt.show()

## Codegen Dependency Analysis

Transitive codegen exposure — how many targets affected by codegen change?


In [ ]:
# --- Codegen Dependency Analysis ---
from build_optimiser.graph import subgraph_for_target

codegen_tgts = target_df[target_df['codegen_file_count'] > 0].copy()
rev_G = G.reverse()

rows = []
for _, row in codegen_tgts.iterrows():
    t = row['cmake_target']
    if t not in G:
        continue
    direct_deps = [n for n in G.predecessors(t) if G[n][t].get('is_direct', False)]
    all_dependants = nx.descendants(rev_G, t)
    transitive_only = all_dependants - set(direct_deps)
    rows.append({
        'cmake_target': t,
        'direct_dependants': len(direct_deps),
        'transitive_dependants': len(transitive_only),
        'total_exposed': len(all_dependants),
        'codegen_compile_time_sum_ms': int(row.get('codegen_compile_time_sum_ms') or 0),
    })

exposure_df = pd.DataFrame(rows).sort_values('total_exposed', ascending=False)

print("=== Transitive Codegen Exposure ===")
display(exposure_df.reset_index(drop=True))

# --- Stacked bar: direct vs transitive dependants ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, max(4, len(exposure_df) * 0.5)))

ax1.barh(exposure_df['cmake_target'], exposure_df['direct_dependants'],
         color='steelblue', label='Direct')
ax1.barh(exposure_df['cmake_target'], exposure_df['transitive_dependants'],
         left=exposure_df['direct_dependants'], color='darkorange', label='Transitive')
ax1.set_xlabel('Exposed targets')
ax1.set_title('Codegen dependency exposure')
ax1.legend(loc='lower right')
ax1.invert_yaxis()

# --- DAG subgraph for highest-exposure codegen target ---
if len(exposure_df) > 0:
    top_target = exposure_df.iloc[0]['cmake_target']
    sub_G = subgraph_for_target(G, top_target, depth=2)

    # Layout: topological depth on x-axis
    topo_depth = {}
    for n in sub_G.nodes():
        depth = 0
        for pred in nx.ancestors(sub_G, n):
            try:
                paths = list(nx.all_simple_paths(sub_G, pred, n))
                for p in paths:
                    depth = max(depth, len(p) - 1)
            except nx.NetworkXError:
                pass
        topo_depth[n] = depth

    max_d = max(topo_depth.values()) if topo_depth else 1
    depth_groups = {}
    for n, d in topo_depth.items():
        depth_groups.setdefault(d, []).append(n)

    pos = {}
    for d, members in depth_groups.items():
        for i, n in enumerate(members):
            pos[n] = (d / max(max_d, 1), (i + 0.5) / len(members))

    codegen_set = set(target_df[target_df['codegen_file_count'] > 0]['cmake_target'])
    node_colors = ['darkorange' if n in codegen_set else 'steelblue' for n in sub_G.nodes()]

    nx.draw_networkx_edges(sub_G, pos, ax=ax2, alpha=0.4, width=0.8,
                           edge_color='grey', arrows=True, arrowsize=10)
    nx.draw_networkx_nodes(sub_G, pos, ax=ax2, node_color=node_colors,
                           node_size=300, alpha=0.9)
    nx.draw_networkx_labels(sub_G, pos, ax=ax2, font_size=7)
    ax2.set_title(f'Dependency neighbourhood of {top_target}')
    ax2.axis('off')

fig.suptitle('Codegen Dependency Analysis', fontsize=14)
plt.tight_layout()
plt.show()

## Recommendations

Caching, parallelism, generated file optimisation, consolidation, isolation.


In [ ]:
# --- Recommendations ---

codegen_tgts = target_df[target_df['codegen_file_count'] > 0].copy()
rev_G = G.reverse()

recommendations = []

for _, row in codegen_tgts.iterrows():
    t = row['cmake_target']
    if t not in G:
        continue

    codegen_ms = int(row.get('codegen_time_ms') or 0)
    cascade = codegen_cascade_cost(G, t, target_df)
    dependant_count = len(nx.descendants(rev_G, t))
    gen_compile_ms = int(row.get('codegen_compile_time_sum_ms') or 0)

    # Caching: high cascade cost → cache codegen outputs
    if cascade > 0:
        msg = (
            f'Cache codegen outputs — cascade cost is '
            f'{cascade:,}ms across {dependant_count} dependants'
        )
        recommendations.append({
            'target': t,
            'category': 'Caching',
            'recommendation': msg,
            'estimated_impact_ms': codegen_ms * dependant_count,
        })

    # Isolation: high fan-out → wrap in interface library
    if dependant_count >= 3:
        msg = (
            f'Wrap codegen output in interface library to '
            f'limit recompilation scope ({dependant_count} dependants)'
        )
        recommendations.append({
            'target': t,
            'category': 'Isolation',
            'recommendation': msg,
            'estimated_impact_ms': gen_compile_ms,
        })

    # File optimisation: high expansion ratio in generated files
    gen_mask = (
        (file_df['cmake_target'] == t)
        & file_df['is_generated']
        & (file_df['compile_time_ms'] > 0)
    )
    gen_file_data = file_df[gen_mask]
    if len(gen_file_data) > 0:
        median_er = gen_file_data['expansion_ratio'].median()
        high_exp = gen_file_data[
            gen_file_data['expansion_ratio'] > median_er * 1.5
        ]
        if len(high_exp) > 0:
            msg = (
                f'{len(high_exp)} generated files with high expansion '
                f'ratio — add forward declarations or reduce includes'
            )
            recommendations.append({
                'target': t,
                'category': 'File optimisation',
                'recommendation': msg,
                'estimated_impact_ms': int(
                    high_exp['compile_time_ms'].sum() * 0.3
                ),
            })

# Parallelism: independent codegen targets
codegen_in_graph = [
    t for t in codegen_tgts['cmake_target'] if t in G
]
independent_pairs = []
for i, a in enumerate(codegen_in_graph):
    for b in codegen_in_graph[i + 1:]:
        if not nx.has_path(G, a, b) and not nx.has_path(G, b, a):
            independent_pairs.append((a, b))

if len(independent_pairs) > 0:
    pair_names = ', '.join(
        f'{a}+{b}' for a, b in independent_pairs[:3]
    )
    total_cg_ms = sum(
        int(target_df[target_df["cmake_target"] == t]
            ["codegen_time_ms"].fillna(0).sum())
        for t in codegen_in_graph
    )
    recommendations.append({
        'target': 'multiple',
        'category': 'Parallelism',
        'recommendation': (
            f'{len(independent_pairs)} independent codegen pairs '
            f'can run in parallel: {pair_names}'
        ),
        'estimated_impact_ms': total_cg_ms // 2,
    })

# Consolidation: multiple small codegen steps
cg_time = codegen_tgts['codegen_time_ms'].fillna(0)
small_codegen = codegen_tgts[cg_time < cg_time.median() * 1.5]
if len(small_codegen) > 1:
    recommendations.append({
        'target': 'multiple',
        'category': 'Consolidation',
        'recommendation': (
            f'{len(small_codegen)} small codegen steps could be '
            f'batched to reduce per-step overhead'
        ),
        'estimated_impact_ms': int(
            small_codegen['codegen_time_ms'].fillna(0).sum() * 0.2
        ),
    })

rec_df = pd.DataFrame(recommendations).sort_values(
    'estimated_impact_ms', ascending=False,
)

print("=== Codegen Optimisation Recommendations ===")
display(rec_df.reset_index(drop=True))

# --- Top 3 actionable recommendations ---
print("\n=== Top 3 Actionable Recommendations ===")
for i, (_, r) in enumerate(rec_df.head(3).iterrows(), 1):
    print(f"\n{i}. [{r['category']}] {r['target']}")
    print(f"   {r['recommendation']}")
    print(f"   Estimated impact: {r['estimated_impact_ms']:,}ms")